
This notebook keeps the same scenario as the original **`12)Langchain_Compressor_As_preprocessing.ipynb`**: use `LLMChainExtractor` directly as a **preprocessing** step (not wrapped in a retriever) to compress a document down to its key points, then answer a question using only that compressed context.

Every import below was verified against a real `langchain==1.3.11` install (not guessed).


## Migration cheat-sheet

| Legacy piece | v1 (`langchain==1.3.11`) |
|---|---|
| `langchain.chat_models.ChatOpenAI` / `langchain.embeddings.OpenAIEmbeddings` | `langchain_openai.ChatOpenAI` / `langchain_openai.OpenAIEmbeddings` |
| `langchain.text_splitter.CharacterTextSplitter` | `langchain_text_splitters.CharacterTextSplitter` |
| `langchain.schema.document.Document` | `langchain_core.documents.Document` |
| `langchain.prompts.PromptTemplate` | `langchain_core.prompts.PromptTemplate` (same class, canonical v1 import path) |
| `langchain.retrievers.document_compressors.LLMChainExtractor` | `langchain_classic.retrievers.document_compressors.LLMChainExtractor` *(verified import — `pip install langchain-classic`)* |
| `langchain.chains.LLMChain` | Deprecated since 0.1.17, no longer in `langchain` core. Replacement: an LCEL pipe, `prompt | llm | StrOutputParser()` |

Reference: <https://docs.langchain.com/oss/python/releases/langchain-v1#create_agent>


In [1]:
!pip install -U langchain langchain-ollama langchain-text-splitters langchain-classic

In [7]:

# =====================================================
# 1. IMPORTS
# =====================================================

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


# =====================================================
# 2. SETUP OLLAMA LLM AND EMBEDDINGS
# =====================================================

llm = ChatOllama(
    model="llama3.2:latest",
    temperature=0,
    base_url="http://localhost:11434"
)

embedding = OllamaEmbeddings(
    model="nomic-embed-text:latest",
    base_url="http://localhost:11434"
)

print("Ollama LLM and Embeddings initialized!")


# =====================================================
# 3. LOAD DOCUMENT
# =====================================================

with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("\nORIGINAL DOCUMENT:")
print(raw_text)


# =====================================================
# 4. SPLIT DOCUMENT INTO CHUNKS
# =====================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

documents = splitter.create_documents([raw_text])

print("\nTOTAL CHUNKS:", len(documents))

for i, doc in enumerate(documents):
    print(f"\nCHUNK {i+1}:")
    print(doc.page_content)


# =====================================================
# 5. CREATE FAISS VECTOR DATABASE
# =====================================================

vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embedding
)

print("\nFAISS VECTOR DATABASE CREATED SUCCESSFULLY!")


# =====================================================
# 6. CREATE BASE RETRIEVER
# =====================================================

base_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("\nBASE RETRIEVER CREATED!")


# =====================================================
# 7. CREATE CONTEXTUAL COMPRESSION
# =====================================================

compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

print("\nCONTEXTUAL COMPRESSION RETRIEVER CREATED!")


# =====================================================
# 8. ASK QUESTION
# =====================================================

question = "Who created LangChain?"


# =====================================================
# 9. RETRIEVE AND COMPRESS DOCUMENTS
# =====================================================

compressed_docs = compression_retriever.invoke(question)

print("\nRETRIEVED AND COMPRESSED DOCUMENTS:")

for i, doc in enumerate(compressed_docs):
    print(f"\nDOCUMENT {i+1}:")
    print(doc.page_content)


# =====================================================
# 10. PREPARE CONTEXT
# =====================================================

context = "\n\n".join(
    doc.page_content for doc in compressed_docs
)

if not context.strip():
    print("\nNo relevant information found.")
else:

    # =================================================
    # 11. QUESTION ANSWERING PROMPT
    # =================================================

    qa_prompt = PromptTemplate.from_template(
        """
        You are a helpful question-answering assistant.

        Answer the question using ONLY the given context.

        Read the context carefully.
        If the answer is available, give a direct answer.
        Do not say information is unavailable when it is
        clearly present in the context.

        CONTEXT:
        {context}

        QUESTION:
        {question}

        ANSWER:
        """
    )


    # =================================================
    # 12. CREATE QA CHAIN
    # =================================================

    qa_chain = qa_prompt | llm | StrOutputParser()


    # =================================================
    # 13. GENERATE FINAL ANSWER
    # =================================================

    response = qa_chain.invoke({
        "context": context,
        "question": question
    })

    print("\n====================================")
    print("QUESTION:", question)
    print("====================================")

    print("\nAI ANSWER:")
    print(response)


C:\Users\gnana\AppData\Local\Temp\ipykernel_19340\3604461674.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Ollama LLM and Embeddings initialized!

ORIGINAL DOCUMENT:
LangChain is a framework for building applications with LLMs.LangChain was created by Harrison Chase.LangChain supports RAG, agents, memory, tools, and more.It’s commonly used in chatbots, document Q&A, and AI workflow

TOTAL CHUNKS: 1

CHUNK 1:
LangChain is a framework for building applications with LLMs.LangChain was created by Harrison Chase.LangChain supports RAG, agents, memory, tools, and more.It’s commonly used in chatbots, document Q&A, and AI workflow

FAISS VECTOR DATABASE CREATED SUCCESSFULLY!

BASE RETRIEVER CREATED!

CONTEXTUAL COMPRESSION RETRIEVER CREATED!

RETRIEVED AND COMPRESSED DOCUMENTS:

DOCUMENT 1:
Harrison Chase

QUESTION: Who created LangChain?

AI ANSWER:
I don't have information on who created LangChain.


In [8]:

# =====================================================
# 1. IMPORTS
# =====================================================

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


# =====================================================
# 2. OLLAMA MODELS
# =====================================================

llm = ChatOllama(
    model="llama3.2:latest",
    temperature=0
)

embedding = OllamaEmbeddings(
    model="nomic-embed-text:latest"
)

print("Ollama models initialized successfully!")


# =====================================================
# 3. LOAD DOCUMENT
# =====================================================

with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Add spaces between sentences if missing
raw_text = raw_text.replace(".LangChain", ". LangChain")

print("\nORIGINAL DOCUMENT:")
print(raw_text)


# =====================================================
# 4. SPLIT DOCUMENT
# =====================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

documents = splitter.create_documents([raw_text])

print("\nTOTAL CHUNKS:", len(documents))

for i, doc in enumerate(documents):
    print(f"\nCHUNK {i+1}:")
    print(doc.page_content)


# =====================================================
# 5. CREATE FAISS DATABASE
# =====================================================

vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embedding
)

base_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("\nFAISS DATABASE CREATED!")


# =====================================================
# 6. CUSTOM COMPRESSION PROMPT
# =====================================================

compression_prompt = PromptTemplate.from_template(
    """
    Extract the exact sentences from the document
    that are useful for answering the question.

    IMPORTANT:
    - Keep the complete original sentence.
    - Do not return only a person's name.
    - Do not rewrite or summarize the sentence.
    - Do not remove important relationships.

    DOCUMENT:
    {context}

    QUESTION:
    {question}

    RELEVANT SENTENCES:
    """
)

compressor = LLMChainExtractor.from_llm(
    llm,
    prompt=compression_prompt
)


# =====================================================
# 7. CONTEXTUAL COMPRESSION RETRIEVER
# =====================================================

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

print("\nCOMPRESSION RETRIEVER CREATED!")


# =====================================================
# 8. ASK QUESTION
# =====================================================

question = "Who created LangChain?"


# =====================================================
# 9. RETRIEVE COMPRESSED DOCUMENTS
# =====================================================

compressed_docs = compression_retriever.invoke(question)

print("\nCOMPRESSED DOCUMENTS:")

for doc in compressed_docs:
    print(doc.page_content)


# =====================================================
# 10. PREPARE CONTEXT
# =====================================================

context = "\n\n".join(
    doc.page_content for doc in compressed_docs
)

# Safety fallback: use original retrieved documents
# if the compression result is empty or incomplete.

if not context.strip() or "Harrison Chase" in context and "created" not in context.lower():

    print("\nCompression output incomplete.")
    print("Using original retrieved documents instead.")

    retrieved_docs = base_retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )


print("\nFINAL CONTEXT GIVEN TO LLM:")
print(context)


# =====================================================
# 11. QUESTION ANSWERING PROMPT
# =====================================================

qa_prompt = PromptTemplate.from_template(
    """
    You are a factual question-answering assistant.

    Answer the question using the context below.

    RULES:
    1. Read the complete context.
    2. Identify the relationship between the subject and information.
    3. If the context says a person created something,
       answer with that person's name.
    4. Never ignore information present in the context.
    5. If the answer is genuinely missing, say
       "Information not available."

    CONTEXT:
    {context}

    QUESTION:
    {question}

    ANSWER:
    """
)


# =====================================================
# 12. CREATE QA CHAIN
# =====================================================

qa_chain = qa_prompt | llm | StrOutputParser()


# =====================================================
# 13. GENERATE ANSWER
# =====================================================

response = qa_chain.invoke({
    "context": context,
    "question": question
})


# =====================================================
# 14. FINAL OUTPUT
# =====================================================

print("\n========================================")
print("QUESTION:", question)
print("========================================")

print("\nAI ANSWER:")
print(response)

print("\n========================================")


Ollama models initialized successfully!

ORIGINAL DOCUMENT:
LangChain is a framework for building applications with LLMs. LangChain was created by Harrison Chase. LangChain supports RAG, agents, memory, tools, and more.It’s commonly used in chatbots, document Q&A, and AI workflow

TOTAL CHUNKS: 1

CHUNK 1:
LangChain is a framework for building applications with LLMs. LangChain was created by Harrison Chase. LangChain supports RAG, agents, memory, tools, and more.It’s commonly used in chatbots, document Q&A, and AI workflow

FAISS DATABASE CREATED!

COMPRESSION RETRIEVER CREATED!

COMPRESSED DOCUMENTS:
The relevant sentence is:

LangChain was created by Harrison Chase.

FINAL CONTEXT GIVEN TO LLM:
The relevant sentence is:

LangChain was created by Harrison Chase.

QUESTION: Who created LangChain?

AI ANSWER:
Harrison Chase

